In [11]:
"""
[최종 정리]
1. 인증 준비 ( PKCE + Authorization URL 생성 )
* code_verifier 제작
* code_challenge 제작
* 인증 URL 생성 -> 브라우저 로그인
* Callback 서버로 Authorization Code 수신

2. Authorization Code -> Access Token + Refresh Token 교환
* Twitter Developer Setting 이 Public Client 로 Secret 사용하지 않음
* 다음 파라미터로 토큰 교환
-> grant_type = authorization_code
-> code = AUTHORIZATION_CODE
-> redirect_uri
-> code_verifier

* 결과를 token.json 에 저장

3. Refresh Token 자동 갱신
* Public Client 규칙
-> Client Secret , Basic Auth 사용 불가.
-> client_id 만 포함

* 갱신된 토큰을 다시 token.json 에 저장

4. Access Token 을 이용한 자동 트윗 업로드


* 실행시 주의 사항
1~5까지 순서대로 실행 후 ( 이떄 localhost:8080 callback 서버 가동 중 )
4번 실행 후 생기는 페이지에서 앱인증 실행

이후 5번 결과에서 나오는 " code= ...... "  <- code 뒷부분을 복사해서
6번 AUTHORIZATION_CODE 에 기입

만약 재실행한다면 token.json 파일 삭제 후 실행
"""



'\n[최종 정리]\n1. 인증 준비 ( PKCE + Authorization URL 생성 )\n* code_verifier 제작\n* code_challenge 제작\n* 인증 URL 생성 -> 브라우저 로그인\n* Callback 서버로 Authorization Code 수신\n\n2. Authorization Code -> Access Token + Refresh Token 교환\n* Twitter Developer Setting 이 Public Client 로 Secret 사용하지 않음\n* 다음 파라미터로 토큰 교환\n-> grant_type = authorization_code\n-> code = AUTHORIZATION_CODE\n-> redirect_uri\n-> code_verifier\n\n* 결과를 token.json 에 저장\n\n3. Refresh Token 자동 갱신\n* Public Client 규칙\n-> Client Secret , Basic Auth 사용 불가.\n-> client_id 만 포함\n\n* 갱신된 토큰을 다시 token.json 에 저장\n\n4. Access Token 을 이용한 자동 트윗 업로드\n\n\n* 실행시 주의 사항 \n1~5까지 순서대로 실행 후 ( 이떄 localhost:8080 callback 서버 가동 중 ) \n4번 실행 후 생기는 페이지에서 앱인증 실행 \n\n이후 5번 결과에서 나오는 " code= ......  <- \n\n'

In [12]:
# [1] 개발자 정보 설정

CLIENT_ID = "CLIENT_ID"  # 중요!!! 테스트 후 꼭 지울 것
CLIENT_SECRET = "CLIENT_SECRET_ID" # 중요!!! 테스트 후 꼭 지울 것
REDIRECT_URL = "http://localhost:8080/callback"
SCOPE = "tweet.read tweet.write users.read offline.access"

In [2]:
# [2] Callback 서버 실행

from http.server import BaseHTTPRequestHandler, HTTPServer
import urllib.parse

class CallbackHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        parsed = urllib.parse.urlparse(self.path)
        params = urllib.parse.parse_qs(parsed.query)

        if "code" in params:
            code = params["code"][0]
            print("Received Authorization Code:", code)

            # 브라우저에 표시
            self.send_response(200)
            self.send_header('Content-type', 'text/html')
            self.end_headers()
            self.wfile.write(b"<h1>Authorization Complete</h1>You can close this tab.")
        else:
            self.send_response(400)
            self.end_headers()
            self.wfile.write(b"Missing authorization code")

def run_server_once():
    server = HTTPServer(('localhost', 8080), CallbackHandler)
    server.timeout = 60  # 1분 대기 후 종료
    print("Listening on http://localhost:8080/callback ...")
    server.handle_request()
    print("Callback server closed")

In [15]:
# [5]

run_server_once()

OSError: [Errno 48] Address already in use

In [13]:
# [3] code_verifier 생성

import os
import hashlib
import base64

def generate_code_verifier():
    return base64.urlsafe_b64encode(os.urandom(40)).rstrip(b'=').decode('utf-8')

def generate_code_challenge(verifier):
    digest = hashlib.sha256(verifier.encode('utf-8')).digest()
    return base64.urlsafe_b64encode(digest).rstrip(b'=').decode('utf-8')

CODE_VERIFIER = generate_code_verifier()
CODE_CHALLENGE = generate_code_challenge(CODE_VERIFIER)

In [14]:
# [4] 인증 URL 만들 때 code_challenge 사용

from urllib.parse import quote

AUTH_URL = (
    "https://twitter.com/i/oauth2/authorize"
    f"?response_type=code"
    f"&client_id={CLIENT_ID}"
    f"&redirect_uri={quote(REDIRECT_URL)}"
    f"&scope={quote(SCOPE)}"
    f"&state=state123"
    f"&code_challenge={CODE_CHALLENGE}"
    f"&code_challenge_method=S256"
)

print(AUTH_URL)

https://twitter.com/i/oauth2/authorize?response_type=code&client_id=CLIENT_ID&redirect_uri=http%3A//localhost%3A8080/callback&scope=tweet.read%20tweet.write%20users.read%20offline.access&state=state123&code_challenge=tWQx-7eZ4IMM9wMfU6iv4bBwEtjSt5nRIVkw8lJeGtk&code_challenge_method=S256


In [16]:
# [6] Authorization Code -> Token 교환

import base64
import requests

AUTHORIZATION_CODE = "AUTHORIZATION_CODE"

def get_token_from_code(code):
    url = "https://api.x.com/2/oauth2/token"

    # ClientID:ClientSecret → Base64 인코딩
    basic_token = base64.b64encode(
        f"{CLIENT_ID}:{CLIENT_SECRET}".encode()
    ).decode()

    headers = {
        "Authorization": f"Basic {basic_token}",
        "Content-Type": "application/x-www-form-urlencoded"
    }

    data = {
        "grant_type": "authorization_code",
        "code": code,
        "redirect_uri": REDIRECT_URL,
        "code_verifier": CODE_VERIFIER,
    }

    response = requests.post(url, data=data, headers=headers, timeout=30)
    return response.json()

token_info = get_token_from_code(AUTHORIZATION_CODE)
token_info

{'error': 'invalid_client',
 'error_description': 'Value passed for the client id was invalid.'}

In [17]:
# [7] 토큰 저장

import json

TOKEN_PATH = "token.json"

def load_token():
    with open(TOKEN_PATH) as f:
        return json.load(f)

def save_token(token_info, code_verifier, client_id):
    data = token_info.copy()
    data["code_verifier"] = code_verifier
    data["client_id"] = client_id

    with open(TOKEN_PATH, "w") as f:
        json.dump(data, f, indent=2)

save_token(token_info, CODE_VERIFIER, CLIENT_ID)

In [18]:
# [8] 토큰 자동 갱신 --> 자동 로그인 [완성]

import json
with open("token.json") as f:
    token_info = json.load(f)
import base64


def refresh_access_token():
    token_info = load_token()

    refresh_token = token_info["refresh_token"]
    code_verifier = token_info["code_verifier"]
    client_id = token_info["client_id"]

    url = "https://api.twitter.com/2/oauth2/token"

    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
    }

    data = {
        "grant_type": "refresh_token",
        "refresh_token": refresh_token,
        "client_id": client_id,
    }

    response = requests.post(url, data=data, headers=headers, timeout=30)
    new_token = response.json()


    # 실패 검사 로직 추가
    if "error" in new_token:
        print("refresh token 갱신 실패:")
        print(json.dumps(new_token, indent=2))
        print("token.json 덮어쓰지 않았습니다.")
        return None

    print(" refresh_token 갱신 성공 ")

    save_token(new_token, code_verifier, client_id)

    return new_token

refresh_access_token()

KeyError: 'refresh_token'

In [19]:
# [9] 갱신된 access_token 으로 트윗 업로드

import requests
import json
with open("token.json") as f:
    token_info = json.load(f)

ACCESS_TOKEN = token_info["access_token"]

url = "https://api.twitter.com/2/tweets"

headers = {
        "Authorization": f"Bearer {ACCESS_TOKEN.strip()}",
        "Content-Type": "application/json"
    }

data = {
    "text": "AURA 서비스 테스트  "
}

response = requests.post(url, headers=headers, json=data, timeout=30)
print(response.status_code)
print(response.json())



KeyError: 'access_token'

In [1]:
# [전체 자동화 버전]

import os
import json
import requests
from urllib.parse import quote
from http.server import BaseHTTPRequestHandler, HTTPServer
import threading
import webbrowser
import base64
import hashlib
import os as _os

TOKEN_PATH = "token.json"
CLIENT_ID = "CLIENT_ID"
REDIRECT_URL = "http://localhost:8080/callback"
SCOPE = "tweet.write users.read tweet.read offline.access"

# PKCE Util

def generate_code_verifier():
    return base64.urlsafe_b64encode(_os.urandom(40)).rstrip(b'=').decode('utf-8')

def generate_code_challenge(verifier):
    digest = hashlib.sha256(verifier.encode('utf-8')).digest()
    return base64.urlsafe_b64encode(digest).rstrip(b'=').decode('utf-8')


# Token Storage

def load_token():
    if not os.path.exists(TOKEN_PATH):
        return None
    with open(TOKEN_PATH) as f:
        return json.load(f)

def save_token(token_info, code_verifier, client_id):
    data = token_info.copy()
    data["code_verifier"] = code_verifier
    data["client_id"] = client_id
    with open(TOKEN_PATH, "w") as f:
        json.dump(data, f, indent=2)


# Callback Server

class CallbackHandler(BaseHTTPRequestHandler):
    authorization_code = None

    def do_GET(self):
        from urllib.parse import urlparse, parse_qs
        parsed = urlparse(self.path)
        params = parse_qs(parsed.query)
        if "code" in params:
            CallbackHandler.authorization_code = params["code"][0]
            self.send_response(200)
            self.send_headers('Content-type', 'text/html')
            self.end_headers()
            self.wfile.write(b"<h1>Authorization Complete</h1>")
        else:
            self.send_response(400)
            self.end_headers()
            self.wfile.write(b"Missing Authorization Code")


# Start Auth Flow

def start_auth_flow():
    code_verifier = generate_code_verifier()
    code_challenge = generate_code_challenge(code_verifier)

    auth_url = (
            "https://twitter.com/i/oauth2/authorize"
            f"?response_type=code"
            f"&client_id={CLIENT_ID}"
            f"&redirect_uri={quote(REDIRECT_URL)}"
            f"&scope={quote(SCOPE)}"
            f"&state=state123"
            f"&code_challenge={code_challenge}"
            f"&code_challenge_method=S256"
    )

    # Start local callback server
    server = HTTPServer(("localhost", 8080), CallbackHandler)
    threading.Thread(target=lambda: server.handle_request()).start()

    webbrowser.open(auth_url)
    print("Waiting for authorization ...")

    while CallbackHandler.authorization_code is None:
        pass

    print("Received code:", CallbackHandler.authorization_code)
    server.server_class()

    return CallbackHandler.authorization_code, code_verifier


# Exchange Code -> Token

def exchange_code_for_token(code, code_verifier):
    url = "https://api.twitter.com/2/oauth2/token"

    headers = {"Content-Type": "application/x-www-form-urlencoded"}

    data = {
        "grant_type": "authorization_code",
        "code": code,
        "redirect_uri": REDIRECT_URL,
        "code_verifier": code_verifier,
        "client_id" : CLIENT_ID,
    }

    response = requests.post(url, headers=headers, data=data)
    return response.json()

# Refresh Token

def refresh_access_token():
    token_info = load_token()
    if token_info is None:
        return None

    url = "https://api.twitter.com/oauth2/token"
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

    data = {
        "grant_type": "refresh_token",
        "refresh_token": token_info["refresh_token"],
        "client_id": token_info["client_id"],
    }

    response = requests.post(url, headers=headers, data=data)
    new_token = response.json()

    if "error" in new_token:
        print("Refresh failed")
        print(new_token)
        return None

    save_token(new_token, token_info["code_verifier"], token_info["client_id"])
    return new_token

# Tweet upload

def upload_tweet(text):
    token_info = load_token()
    access_token = token_info["access_token"]

    url = "https://api.twitter.com/2/tweets"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }
    data = {"text": text}

    response = requests.post(url, headers=headers, json=data)
    print(response.status_code)
    print(response.json())

# Full Flow

def run_full_flow():
    token_info = load_token()

    if token_info is None:
        print("No token found -> Starting authentication flow")
        code, code_verifier = start_auth_flow()
        token_info = exchange_code_for_token(code, code_verifier)
        save_token(token_info, code_verifier, CLIENT_ID)
    else:
        print("Token found -> Trying refresh")
        new_token = refresh_access_token()
        if new_token is None:
            print("Refresh failed -> Re-authenticating")
            os.remove(TOKEN_PATH)
            return run_full_flow()

    print("Uploading test tweet")
    upload_tweet("AURA 자동화 테스트")


if __name__ == "__main__":
    run_full_flow()

No token found -> Starting authentication flow


AttributeError: module 'base64' has no attribute 'urlsafe_b53encode'